### Capa Silver (Limpieza y Estandarización)
Esta capa procesa los datos crudos de la capa Bronze aplicando reglas de calidad y transformaciones, para luego almacenarlos en formato Delta Lake.
Las operaciones incluyen:
- Filtrado de PK nulas y montos no positivos.
- Deduplicación de transacciones y clientes.
- Estandarización de textos (espacios, mayúsculas).
- Conversión de fechas.
- Upsert (MERGE) para transacciones en `transacciones_plata_robert` y para clientes en `clientes_plata_robert`.


In [0]:
from pyspark.sql.functions import col, trim, upper, round, regexp_replace, coalesce, to_timestamp, to_date, date_format, try_to_timestamp, expr
from delta.tables import DeltaTable


In [0]:
print("1. Leyendo datos desde la Capa Bronze (Delta Lake)...")
df_tx_bronce = spark.table("workspace.aml_proyect.transacciones_bronce")
df_clientes_bronce = spark.table("workspace.aml_proyect.clientes_bronce")


In [0]:
print("2. Aplicando reglas de calidad (Capa Plata)...")
# Limpieza de Transacciones
df_tx_silver = (df_tx_bronce
    .filter(col("PK_Transaccion").isNotNull())
    .withColumn("Monto_Original", round(col("Monto_Original").cast("double"), 2))
    .withColumn("Monto_USD", round(col("Monto_USD").cast("double"), 2))
    .filter(col("Monto_USD") > 0)
    .dropDuplicates(["PK_Transaccion"])
    .withColumn("FK_Canal", trim(col("FK_Canal")))
    .withColumn("Moneda", upper(trim(col("Moneda"))))
    .withColumn("Tiempo_Real", expr("coalesce(try_to_timestamp(FK_Tiempo, 'yyyy-MM-dd HH:mm:ss'), try_to_timestamp(FK_Tiempo, 'dd/MM/yyyy HH:mm:ss'))"))
    .withColumn("Fecha_Transaccion", to_date(col("Tiempo_Real")))
    .withColumn("Hora_Transaccion", date_format(col("Tiempo_Real"), "HH:mm:ss"))
    .drop("FK_Tiempo", "Tiempo_Real")
)

# Limpieza de Clientes
df_clientes_silver = (df_clientes_bronce
    .filter(col("PK_Cliente").isNotNull())
    .dropDuplicates(["PK_Cliente"])
    .withColumn("Ocupacion_Declarada", trim(col("Ocupacion_Declarada")))
)


In [0]:
print("3. Ejecutando MERGE (Upsert) en tablas Delta...")

def upsert_delta(df_source, table_name, join_condition):
    if spark.catalog.tableExists(table_name):
        target_table = DeltaTable.forName(spark, table_name)
        target_table.alias("destino").merge(
            df_source.alias("origen"),
            join_condition
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        df_source.write.format("delta").mode("overwrite").saveAsTable(table_name)

# Transacciones
upsert_delta(
    df_source=df_tx_silver,
    table_name="workspace.aml_proyect.transacciones_plata_robert",
    join_condition="destino.PK_Transaccion = origen.PK_Transaccion"
)

# Clientes
upsert_delta(
    df_source=df_clientes_silver,
    table_name="workspace.aml_proyect.clientes_plata_robert",
    join_condition="destino.PK_Cliente = origen.PK_Cliente"
)

print("Datos procesados exitosamente en la capa Silver.")
display(spark.sql("SELECT * FROM workspace.aml_proyect.transacciones_plata_robert LIMIT 5"))
